# Review Stooq Anomalies

Aggregated to **(ticker, period, rule)** where `period` is the calendar year as a string (e.g. `'2024'`). Per-bar review is infeasible for a 45M-row table; per-year is the natural granularity.

Reviews stored in `data/data_quality/stooq_anomaly_reviews.toml`. Rules in `src/irp/quality/stooq_rules.py`.

In [1]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, HTML

from irp.quality.stooq_runner import run
from irp.quality.stooq_inspect import inspect, inspect_long, flagged_bars, yahoo_bars, yahoo_url, staleness_scan, load_staleness_results, load_yahoo_errors
from irp.quality.stooq_reviews import add_review, load_reviews_df

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 240)


def _fmt_num(x):
    if pd.isna(x):
        return ''
    try:
        return f'{float(x):,.4f}'
    except (TypeError, ValueError):
        return str(x)

In [2]:
all_findings = run(skip_reviewed=True)
done: set[tuple[str, str, str]] = set()
state = {'idx': 0, 'sort': 'count_desc', 'min_count': 0, 'rules': None, 'markets': None}

_rule_options = sorted(all_findings['Rule'].unique()) if len(all_findings) else []
_market_options = (
    sorted({m for s in all_findings['Market'].dropna() for m in s.split(' / ') if m})
    if len(all_findings) else []
)

sort_w = widgets.Dropdown(
    options=['count_desc', 'ticker', 'rule'],
    value='count_desc',
    description='Sort:',
)
min_count_w = widgets.IntText(value=0, description='Min count:', layout=widgets.Layout(width='160px'))
rules_w = widgets.SelectMultiple(
    options=_rule_options,
    description='Rules:',
    rows=min(6, max(1, len(_rule_options))),
    layout=widgets.Layout(width='400px'),
)
markets_w = widgets.SelectMultiple(
    options=_market_options,
    description='Markets:',
    rows=min(8, max(1, len(_market_options))),
    layout=widgets.Layout(width='400px'),
)
apply_btn = widgets.Button(description='Apply', button_style='info')

print(f'{len(all_findings)} unreviewed findings loaded')

159 unreviewed findings loaded


In [3]:
def _key(row):
    return (row['Ticker'], row['Period_str'], row['Rule'])


def _market_match(market_str: str, selected: set[str]) -> bool:
    if not market_str:
        return False
    return bool(set(market_str.split(' / ')) & selected)


def _rebuild_queue():
    df = all_findings
    if done:
        df = df[[_key(r) not in done for _, r in df.iterrows()]]
    if state['rules']:
        df = df[df['Rule'].isin(state['rules'])]
    if state['markets']:
        sel = set(state['markets'])
        df = df[df['Market'].fillna('').apply(lambda m: _market_match(m, sel))]
    if state['min_count'] > 0:
        df = df[df['count'] >= state['min_count']]
    if state['sort'] == 'count_desc':
        df = df.sort_values('count', ascending=False)
    elif state['sort'] == 'ticker':
        df = df.sort_values(['Ticker', 'Year', 'Rule'], ascending=[True, False, True])
    elif state['sort'] == 'rule':
        df = df.sort_values(['Rule', 'Ticker', 'Year'], ascending=[True, True, False])
    return df.reset_index(drop=True)


def _siblings(current_row, queue):
    t, ps, rule = _key(current_row)
    same_rule = queue[
        (queue['Ticker'] == t) & (queue['Rule'] == rule) & (queue['Period_str'] != ps)
    ]
    same_period = queue[
        (queue['Ticker'] == t) & (queue['Period_str'] == ps) & (queue['Rule'] != rule)
    ]
    return same_rule, same_period


def _make_cb_row(row, ticked):
    label = f"{row['Ticker']} {row['Period_str']}  {row['Rule']}  ({int(row['count']):,} bars)"
    cb = widgets.Checkbox(value=ticked, description=label, indent=False,
                          layout=widgets.Layout(width='600px'))
    return widgets.HBox([cb]), cb


def _render_queue_list(max_rows: int = 50):
    if len(queue) == 0:
        queue_list_html.value = '<i>No unreviewed findings for current filter.</i>'
        return
    show = queue[['Ticker', 'Period_str', 'Rule', 'count', 'Market']].head(max_rows).copy()
    show.insert(0, '', ['→' if i == state['idx'] else '' for i in range(len(show))])
    show['count'] = show['count'].map(lambda x: f'{int(x):,}')
    header = f'<p><b>{len(queue)} unreviewed</b>'
    if len(queue) > max_rows:
        header += f' (showing top {max_rows})'
    header += '</p>'
    queue_list_html.value = header + show.to_html(index=False, escape=False)


def _ohlcv_compare(ticker: str, sample_dates: list[int]) -> pd.DataFrame:
    """Merge stooq + yahoo OHLCV for the flagged dates into one table."""
    s = flagged_bars(ticker, sample_dates)
    y = yahoo_bars(ticker, sample_dates)
    if s.empty and y.empty:
        return pd.DataFrame()
    s2 = s.rename(columns={'O': 'O_st', 'H': 'H_st', 'L': 'L_st', 'C': 'C_st', 'V': 'V_st'}) if len(s) else pd.DataFrame(columns=['Date'])
    y2 = y.rename(columns={'O': 'O_yh', 'H': 'H_yh', 'L': 'L_yh', 'C': 'C_yh', 'V': 'V_yh'}) if len(y) else pd.DataFrame(columns=['Date'])
    return s2.merge(y2, on='Date', how='outer').sort_values('Date').reset_index(drop=True)


def render():
    global queue
    if state['idx'] >= len(queue):
        container.children = (widgets.HTML('<i>Queue empty for current filter.</i>'),)
        sibling_state['checkboxes'] = []
        _render_queue_list()
        return
    row = queue.iloc[state['idx']]
    progress = f'{state["idx"] + 1}/{len(queue)}'
    market_str = row.get('Market') or '(no market)'
    sample_dates = list(row['sample_dates']) if row['sample_dates'] is not None else []
    yurl = yahoo_url(row['Ticker'], row['Period_str'])
    children = [
        widgets.HTML(
            f'<h3>[{progress}] {row["Ticker"]} {row["Period_str"]} — {row["Rule"]}</h3>'
            f'<p>{int(row["count"]):,} bad bars &nbsp;|&nbsp; Market: {market_str} &nbsp;|&nbsp; '
            f'<a href="{yurl}" target="_blank">Yahoo Finance</a></p>'
        )
    ]

    cmp_df = _ohlcv_compare(row['Ticker'], sample_dates)
    if len(cmp_df):
        children.append(widgets.HTML('<b>Flagged bars — Stooq (_st) vs Yahoo (_yh):</b>'))
        num_cols = [c for c in cmp_df.columns if c != 'Date']
        children.append(widgets.HTML(
            cmp_df.style.format({c: _fmt_num for c in num_cols}).to_html()
        ))

    fig_short = inspect(row['Rule'], row['Ticker'], row['Period_str'], sample_dates)
    fig_long = inspect_long(row['Ticker'], row['Period_str'], sample_dates)
    chart_widgets = []
    for fig in (fig_short, fig_long):
        if fig is None:
            continue
        out = widgets.Output()
        with out:
            fig.show()
        chart_widgets.append(out)
    if chart_widgets:
        children.append(widgets.HBox(chart_widgets))

    same_rule, same_period = _siblings(row, queue)
    cbs = []
    if len(same_rule):
        children.append(widgets.HTML(f'<b>Same rule, other periods ({len(same_rule)}) — pre-ticked:</b>'))
        for _, sib in same_rule.iterrows():
            hbox, cb = _make_cb_row(sib, ticked=True)
            cbs.append((cb, sib))
            children.append(hbox)
    if len(same_period):
        children.append(widgets.HTML(f'<b>Same period, other rules ({len(same_period)}) — unticked:</b>'))
        for _, sib in same_period.iterrows():
            hbox, cb = _make_cb_row(sib, ticked=False)
            cbs.append((cb, sib))
            children.append(hbox)

    container.children = tuple(children)
    sibling_state['checkboxes'] = cbs
    note_w.value = ''
    _render_queue_list()


def on_next(_):
    global queue
    if state['idx'] >= len(queue):
        return
    row = queue.iloc[state['idx']]
    targets = [row]
    for cb, sib in sibling_state['checkboxes']:
        if cb.value:
            targets.append(sib)
    for r in targets:
        add_review(r['Ticker'], r['Period_str'], r['Rule'], status_w.value, note_w.value)
        done.add(_key(r))
    queue = _rebuild_queue()
    render()


def on_skip(_):
    state['idx'] += 1
    render()


def on_apply(_):
    global queue
    state['sort'] = sort_w.value
    state['min_count'] = int(min_count_w.value)
    state['rules'] = list(rules_w.value) or None
    state['markets'] = list(markets_w.value) or None
    queue = _rebuild_queue()
    state['idx'] = 0
    render()


queue = _rebuild_queue()
container = widgets.VBox([])
queue_list_html = widgets.HTML()
note_w = widgets.Textarea(description='Note:', layout=widgets.Layout(width='800px', height='80px'))
status_w = widgets.Dropdown(options=['ok', 'data_error', 'to_check'], description='Status:', value='ok')
next_btn = widgets.Button(description='Mark Reviewed & Next', button_style='primary')
skip_btn = widgets.Button(description='Skip')
sibling_state = {'checkboxes': []}
next_btn.on_click(on_next)
skip_btn.on_click(on_skip)
apply_btn.on_click(on_apply)
render()

HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: 1YCHY"}}}
$1YCHY: possibly delisted; no timezone found

1 Failed download:
['1YCHY']: possibly delisted; no timezone found


## Queue

In [4]:
display(widgets.HBox([sort_w, min_count_w]))
display(widgets.HBox([rules_w, markets_w]))
display(apply_btn)
display(queue_list_html)

Button(button_style='info', description='Apply', style=ButtonStyle())

HTML(value='<p><b>159 unreviewed</b> (showing top 50)</p><table border="1" class="dataframe">\n  <thead>\n    …

## Reviewer

In [5]:
display(widgets.VBox([
    container,
    note_w,
    status_w,
    widgets.HBox([next_btn, skip_btn]),
]))

## Staleness Scan

Compare Stooq close vs Yahoo adjusted close at 1-, 3-, 5-year-back snapshots.
. Ratio > 1 → Stooq missing post-snapshot dividend back-adjustments.

In [ ]:
# Resume-safe: re-running skips already-checked tickers.
# Delete data/data_quality/staleness_scan.csv to start over.
scan_df = staleness_scan()
print(f"{len(scan_df)} findings above threshold | {scan_df['Ticker'].nunique()} tickers affected")
scan_df

HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: PSEC_A"}}}
$PSEC_A: possibly delisted; no timezone found
$ABR_E: possibly delisted; no timezone found
$RAPT: possibly delisted; no timezone found
$^_PL: possibly delisted; no timezone found
$CMS_B: possibly delisted; no timezone found
$PMT_C: possibly delisted; no timezone found
$DUK_A: possibly delisted; no timezone found
$OAK_B: possibly delisted; no timezone found
$DLR_L: possibly delisted; no timezone found
$TWO_A: possibly delisted; no timezone found
$AXS_E: possibly delisted; no timezone found
$PSA_M: possibly delisted; no timezone found
$AGM_E: possibly delisted; no timezone found
$ONFOW: possibly delisted; no price data found  (1d 2023-05-14 -> 2023-05-22) (Yahoo error = "Data doesn't exist for startDate = 1684036800, endDate = 1684728000")
$BZFDW: possibly delisted; no price data found  (1d 2023-05-14 -> 2023-05-22) (Yahoo error = "Data doesn't exist for startD

In [ ]:
mask = scan_df['Ticker']=='AMZN'
scan_df[mask]

## Audit

In [ ]:
_reviews = load_reviews_df()
print(f'{len(_reviews)} reviews recorded')
display(_reviews)